In [ ]:
import time
import re
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import zipfile
from pathlib import Path
import geopandas as gpd
import folium
import branca.colormap as cm




# This scraper is selenium based and accounts for the user manually closing a popup and returning to the script.
# It iterates through the table and clicks each section

url = "https://www.opensecrets.org/dark-money/top-elections"

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

all_rows = []

try:
    driver.get(url)
    time.sleep(5)

    wait.until(lambda d: d.find_elements(By.CSS_SELECTOR, "table.DataTable tbody tr"))

    page = 1

    while True:
        print(f"Scraping page {page}...")

        soup = BeautifulSoup(driver.page_source, "html.parser")
        table = soup.select_one("table.DataTable")

        rows = table.select("tbody tr")

        for row in rows:
            cells = row.select("td")

            if not cells:
                continue

            race_cell = cells[0]
            link = race_cell.select_one("a")

            race = race_cell.get_text(" ", strip=True)

            race_id = None
            if link and link.get("href"):
                href = link.get("href")
                query = parse_qs(urlparse(href).query)
                race_id = query.get("id", [None])[0]

            values = [td.get_text(" ", strip=True) for td in cells]

            all_rows.append({
                "race_id": race_id,
                "Race": race,
                "Total": values[1],
                "For Dems": values[2],
                "Against Dems": values[3],
                "For Repubs": values[4],
                "Against Repubs": values[5],
            })

        next_buttons = driver.find_elements(By.CSS_SELECTOR, ".dataTables_paginate .next")

        if not next_buttons:
            break

        next_button = next_buttons[0]

        if "disabled" in next_button.get_attribute("class"):
            break

        old_first_row = driver.find_element(By.CSS_SELECTOR, "table.DataTable tbody tr").text

        driver.execute_script("arguments[0].click();", next_button)

        wait.until(
            lambda d: d.find_element(By.CSS_SELECTOR, "table.DataTable tbody tr").text != old_first_row
        )

        time.sleep(1)
        page += 1

finally:
    driver.quit()


df = pd.DataFrame(all_rows).drop_duplicates().reset_index(drop=True)


df.to_csv("opensecrets_dark_money.csv", index=False)

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
